In [2]:
#start session
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("wheat_futures_price_prediction").getOrCreate()

25/04/21 13:24:09 WARN Utils: Your hostname, Presleys-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.1.134.32 instead (on interface en0)
25/04/21 13:24:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/21 13:24:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


25/04/21 13:24:24 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [ ]:
## Importing Required Libraries
from pyspark.sql.functions import col, isnan, when, count
from pyspark.sql.functions import to_date, col
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt



#Read weather and pricing data
weather_raw = spark.read.csv('weather_data_RAW.csv', header=True, inferSchema=True)
wheat_price_data = spark.read.csv('US_wheat_future_pricing_RAW.csv', header=True, inferSchema=True)

#Parse date columns
weather = weather_raw.withColumn(
    "Date",
    to_date(col("datetime"), "M/d/yyyy")      
).drop("datetime")                            

pricing = wheat_price_data.withColumn(
    "Date",
    to_date(col("Date"), "M/d/yyyy")        
)

#Join on Date
merged = weather.join(pricing, on="Date", how="left")

merged.show(10, truncate=False)

#strip column names
old_cols = merged.columns
new_cols = [c.replace('.', '').replace(' ', '_') for c in old_cols]
merged_clean = merged.toDF(*new_cols)

#check for nulls
null_exprs = []
for c, dtype in merged_clean.dtypes:
    cond = col(c).isNull() | (isnan(col(c)) if dtype in ("double","float") else col(c).isNull())
    null_exprs.append(count(when(cond, col(c))).alias(f"{c}_missing"))

merged_clean.select(*null_exprs).show(truncate=False)

#Summary statistics
merged_clean.describe().show(truncate=False)

+----------+-----+-------+-------+----+------------+------------+---------+----+--------+------+----------+-----------+----------+----+---------+--------+---------+-------+----------------+----------+----------+--------------+-----------+-------+----------+-------------------+-------------------+---------+----------------------------+--------------------------------------------------------------------------------+-----------------+----------------------------------------------------------+------+------+------+------+------+--------+
|Date      |name |tempmax|tempmin|temp|feelslikemax|feelslikemin|feelslike|dew |humidity|precip|precipprob|precipcover|preciptype|snow|snowdepth|windgust|windspeed|winddir|sealevelpressure|cloudcover|visibility|solarradiation|solarenergy|uvindex|severerisk|sunrise            |sunset             |moonphase|conditions                  |description                                                                     |icon             |stations                  

In [16]:
merged_clean.write.csv('merged_clean.csv', header=True, mode='overwrite')

In [21]:
visualsDF = pd.read_csv('merged_CLEAN.csv')


In [ ]:
visualsDF.describe()
#Check for null values
for column in visualsDF.columns:
    if visualsDF[column].isnull().sum() > 0:
        print(f"The column '{column}' has {visualsDF[column].isnull().sum()} null values.")
    else:
        pass
visualsDF.info()


The column 'preciptype' has 7951 null values.
The column 'windgust' has 3 null values.
The column 'visibility' has 343 null values.
The column 'severerisk' has 9844 null values.
The column 'Price' has 4376 null values.
The column 'Open' has 4376 null values.
The column 'High' has 4376 null values.
The column 'Low' has 4376 null values.
The column 'Vol' has 4552 null values.
The column 'Change_%' has 4376 null values.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14616 entries, 0 to 14615
Data columns (total 39 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Date              14616 non-null  object 
 1   name              14616 non-null  int64  
 2   tempmax           14616 non-null  float64
 3   tempmin           14616 non-null  float64
 4   temp              14616 non-null  float64
 5   feelslikemax      14616 non-null  float64
 6   feelslikemin      14616 non-null  float64
 7   feelslike         14616 non-null  float64


Index(['Date', 'preciptype', 'sunrise', 'sunset', 'conditions', 'description',
       'icon', 'stations', 'Price', 'Open', 'High', 'Low', 'Vol', 'Change_%'],
      dtype='object')


25/04/21 21:01:19 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 978870 ms exceeds timeout 120000 ms
25/04/21 21:01:19 WARN SparkContext: Killing executors is not supported by current scheduler.
25/04/21 21:01:23 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [27]:


# Select relevant columns
selected_columns = [
    'tempmax', 'tempmin', 'temp', 'feelslikemax', 'feelslikemin', 'feelslike',
    'dew', 'humidity', 'precip', 'precipprob', 'precipcover', 'snow',
    'snowdepth', 'windgust', 'windspeed', 'winddir', 'sealevelpressure',
    'cloudcover', 'visibility', 'solarradiation', 'solarenergy', 'uvindex',
    'severerisk', 'Open', 'High', 'Low', 'Vol', 'Price'
]

# Filter DataFrame
cor_df = visualsDF[selected_columns].copy()

# Compute correlation
corr = cor_df.corr()

# Plot heatmap
plt.figure(figsize=(16, 12))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap of Influential Variables')
plt.show()


ValueError: could not convert string to float: '1,009.25'